In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("/content/cleaned_dataset (4).csv")

In [3]:
df.head()

,Unnamed: 0,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,...,device_type_mobile,device_type_web,pay_NetBanking,pay_UPI,pay_Wallet,is_balance_missing,balance_utilization,transaction_status_pending,transaction_status_success,is_ip_suspicious
0,0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14 11:40:15.672605,Jaipur,Ahmedabad,Travel,DEV-C894B8F5,mobile,...,1.0,0.0,0,0,0,0,0.053180,0,1,0
1,1,TXN-84B8FC890498,USR0043,308.516781,2024-06-02 09:52:57.000000,Delhi,Delhi,Travel,DEV-EC8BBC24,mobile,...,1.0,0.0,0,1,0,0,0.002518,0,1,0
2,2,TXN-95B71538FFAD,USR0044,3159.000000,2024-03-18 21:09:35.000000,Jaipur,Jaipur,Unknown,D??,web,...,0.0,1.0,0,0,0,0,0.018702,0,1,0
3,3,TXN-657F2702B5B2,USR0060,5349.730000,2024-07-02 03:50:23.000000,Bangalore,Bangalore,Education,DEV-888653EA,web,...,0.0,1.0,0,1,0,0,0.591391,0,1,0
4,4,TXN-84C8BBF69256,USR0020,2454.565213,2024-03-13 23:25:00.000000,Jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,...,1.0,0.0,0,0,0,0,0.016033,0,1,0


In [4]:
df.drop("Unnamed: 0",axis=1,inplace=True)

In [5]:
df.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,account_balance,...,device_type_mobile,device_type_web,pay_NetBanking,pay_UPI,pay_Wallet,is_balance_missing,balance_utilization,transaction_status_pending,transaction_status_success,is_ip_suspicious
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14 11:40:15.672605,Jaipur,Ahmedabad,Travel,DEV-C894B8F5,mobile,140227.79,...,1.0,0.0,0,0,0,0,0.053180,0,1,0
1,TXN-84B8FC890498,USR0043,308.516781,2024-06-02 09:52:57.000000,Delhi,Delhi,Travel,DEV-EC8BBC24,mobile,122548.77,...,1.0,0.0,0,1,0,0,0.002518,0,1,0
2,TXN-95B71538FFAD,USR0044,3159.000000,2024-03-18 21:09:35.000000,Jaipur,Jaipur,Unknown,D??,web,168910.49,...,0.0,1.0,0,0,0,0,0.018702,0,1,0
3,TXN-657F2702B5B2,USR0060,5349.730000,2024-07-02 03:50:23.000000,Bangalore,Bangalore,Education,DEV-888653EA,web,9046.00,...,0.0,1.0,0,1,0,0,0.591391,0,1,0
4,TXN-84C8BBF69256,USR0020,2454.565213,2024-03-13 23:25:00.000000,Jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,153095.35,...,1.0,0.0,0,0,0,0,0.016033,0,1,0


In [6]:
df.columns

Index(['transaction_id', 'user_id', 'transaction_amount',
       'transaction_timestamp', 'user_location', 'merchant_location',
       'merchant_category', 'device_id', 'device_type', 'account_balance',
       'ip_address', 'is_duplicate_txn', 'user_transaction_count',
       'is_amount_missing', 'is_amount_outlier', 'txn_month', 'txn_day',
       'txn_hour', 'txn_minute', 'txn_day_of_week', 'txn_is_weekend',
       'user_location_lat', 'user_location_lon', 'merchant_location_lat',
       'merchant_location_lon', 'user_category_affinity',
       'category_economic_weight', 'unique_users_on_device',
       'is_unique_users_on_device_outlier', 'device_type_mobile',
       'device_type_web', 'pay_NetBanking', 'pay_UPI', 'pay_Wallet',
       'is_balance_missing', 'balance_utilization',
       'transaction_status_pending', 'transaction_status_success',
       'is_ip_suspicious'],
      dtype='object')

In [7]:
# If is_duplicate_txn is 1, is_fraud becomes 1. Otherwise, it stays 0.
df['is_fraud'] = np.where(df['is_duplicate_txn'] == 1, 1, 0)

In [8]:
df["is_duplicate_txn"].value_counts()

,count
is_duplicate_txn,
0,1426


In [9]:
df["is_fraud"]=np.where(df["is_ip_suspicious"]==1,1,0)

In [10]:
df["is_ip_suspicious"].value_counts()

,count
is_ip_suspicious,
0,1323
1,103


In [11]:
threshold=df['balance_utilization'].quantile(0.99)
df["is_fraud"]=np.where(df["balance_utilization"]>threshold,1,0)

In [12]:
df["is_fraud"].value_counts()

,count
is_fraud,
0,1411
1,15


In [13]:
df["is_fraud"]=np.where(df['unique_users_on_device']>=3,1,0)

In [14]:
df["is_fraud"].value_counts()

,count
is_fraud,
0,1420
1,6


In [15]:
# 7. The Night Owl
# FLAG: Transaction between 1 AM - 5 AM AND (amount > 25000 OR amount < 25)
df['is_fraud'] = (
    (df['txn_hour'] >= 1) &
    (df['txn_hour'] <= 5) &
    ((df['transaction_amount'] > 100000) | (df['transaction_amount'] < 25)) # <-- Notice the extra outer parentheses here!
).astype(int)

In [16]:
df["is_fraud"].value_counts()

,count
is_fraud,
0,1422
1,4


In [17]:
# 1. First, create individual pattern columns so you don't lose work
# This makes your dataset "explainable" for the judges

# Pattern: Suspicious IP
df['pattern_ip'] = np.where(df['is_ip_suspicious'] == 1, 1, 0)

# Pattern: Extreme Balance Utilization (Top 1%)
threshold_bal = df['balance_utilization'].quantile(0.99)
df['pattern_balance'] = np.where(df['balance_utilization'] > threshold_bal, 1, 0)

# Pattern: Fraud Ring (Multiple users on one device)
df['pattern_fraud_ring'] = np.where(df['unique_users_on_device'] >= 3, 1, 0)

# Pattern: Night Owl (Late night + extreme amounts)
df['pattern_night_owl'] = (
    (df['txn_hour'] >= 1) &
    (df['txn_hour'] <= 5) &
    ((df['transaction_amount'] > 100000) | (df['transaction_amount'] < 25))
).astype(int)


df['unique_users_on_ip'] = df.groupby('ip_address')['user_id'].transform('nunique')
df['pattern_botnet'] = (df['unique_users_on_ip'] > 3).astype(int)

# =====================================================================
# 2. CREATE THE MASTER IS_FRAUD TARGET
# =====================================================================
# Identify all columns we just created that start with 'pattern_'
pattern_cols = [col for col in df.columns if col.startswith('pattern_')]

# If ANY pattern is 1, then is_fraud is 1. We use .max(axis=1) for this.
df['is_fraud'] = df[pattern_cols].max(axis=1)

# 3. View the Results
print("=== FINAL FRAUD COUNTS ===")
print(df['is_fraud'].value_counts())
print("\nBreakdown by Pattern:")
for col in pattern_cols:
    print(f"{col}: {df[col].sum()}")

=== FINAL FRAUD COUNTS ===
is_fraud
0    1299
1     127
Name: count, dtype: int64

Breakdown by Pattern:
pattern_ip: 103
pattern_balance: 15
pattern_fraud_ring: 6
pattern_night_owl: 4
pattern_botnet: 20


In [18]:
# 2. Ensure transaction_timestamp is a datetime object
df['transaction_timestamp'] = pd.to_datetime(df['transaction_timestamp'])

# 3. Sort by user_id and timestamp for accurate chronological windowing
df = df.sort_values(by=['user_id', 'transaction_timestamp'])

# --- Set your Fraud Rules Here ---
# FIX: In newer pandas versions, use lowercase 'h' instead of 'H'
TIME_FRAME = '1h'          # e.g., '1h' for 1 hour, '10min' for 10 minutes, '1d' for 1 day
TRANSACTION_THRESHOLD = 3  # Flag if a user exceeds 3 transactions within the TIME_FRAME
# ---------------------------------

# 4. Set timestamp as the index temporarily (required for pandas time-based rolling windows)
df = df.set_index('transaction_timestamp')

# 5. Group by user_id and count transactions within the rolling time frame
df['txn_count_in_window'] = df.groupby('user_id')['transaction_id'].transform(
    lambda x: x.rolling(TIME_FRAME).count()
)

# 6. Create the fraud flag condition (1 if True/Fraud, 0 if False/Normal)
df['is_velocity_fraud'] = (df['txn_count_in_window'] >= TRANSACTION_THRESHOLD).astype(int)

# 7. Reset the index to put transaction_timestamp back as a regular column
df = df.reset_index()

# Check how many transactions were flagged
num_flagged = df['is_velocity_fraud'].sum()
print(f"Number of transactions flagged as fraud: {num_flagged}")

# Preview the flagged transactions
display(df[df['is_velocity_fraud'] == 1][['user_id', 'transaction_timestamp', 'txn_count_in_window', 'is_velocity_fraud']].head())

Number of transactions flagged as fraud: 13


,user_id,transaction_timestamp,txn_count_in_window,is_velocity_fraud
108,USR0007,2024-03-26 03:47:40,3.0,1
239,USR0015,2024-01-17 04:38:07,3.0,1
413,USR0023,2024-02-19 22:43:13,3.0,1
424,USR0024,2024-01-28 03:36:57,3.0,1
492,USR0028,2024-01-28 10:40:40,3.0,1


In [19]:
import pandas as pd
import numpy as np

def engineer_fraud_features(df):
    """
    Applies all fraud detection rules to the dataframe and creates a master is_fraud label.
    """
    # 0. Setup: Ensure timestamp is datetime for velocity calculations
    if not pd.api.types.is_datetime64_any_dtype(df['transaction_timestamp']):
        df['transaction_timestamp'] = pd.to_datetime(df['transaction_timestamp'])

    # 1. Pattern: Duplicate Transaction
    # (Rescued from Cell 7 before it was overwritten!)
    df['pattern_duplicate'] = np.where(df['is_duplicate_txn'] == 1, 1, 0)

    # 2. Pattern: Suspicious IP
    df['pattern_ip'] = np.where(df['is_ip_suspicious'] == 1, 1, 0)

    # 3. Pattern: Extreme Balance Utilization (Top 1%)
    threshold_bal = df['balance_utilization'].quantile(0.99)
    df['pattern_balance'] = np.where(df['balance_utilization'] > threshold_bal, 1, 0)

    # 4. Pattern: Fraud Ring (Multiple users on one device)
    df['pattern_fraud_ring'] = np.where(df['unique_users_on_device'] >= 3, 1, 0)

    # 5. Pattern: Night Owl (Late night + extreme amounts)
    df['pattern_night_owl'] = (
        (df['txn_hour'] >= 1) &
        (df['txn_hour'] <= 5) &
        ((df['transaction_amount'] > 100000) | (df['transaction_amount'] < 25))
    ).astype(int)

    # 6. Pattern: Botnet (Multiple unique users on same IP)
    df['unique_users_on_ip'] = df.groupby('ip_address')['user_id'].transform('nunique')
    df['pattern_botnet'] = (df['unique_users_on_ip'] > 3).astype(int)

    # 7. Pattern: Velocity Fraud (High frequency in short time)
    # Sort values and temporarily set index for time-based rolling window
    df = df.sort_values(by=['user_id', 'transaction_timestamp'])
    df = df.set_index('transaction_timestamp')

    # Calculate transactions in a 1-hour window per user
    df['txn_count_in_window'] = df.groupby('user_id')['transaction_id'].transform(
        lambda x: x.rolling('1h').count()
    )
    df['pattern_velocity'] = (df['txn_count_in_window'] >= 3).astype(int)

    # Reset index to bring transaction_timestamp back as a regular column
    df = df.reset_index()

    # =====================================================================
    # MASTER IS_FRAUD TARGET
    # =====================================================================
    # Dynamically grab all columns we just created that start with 'pattern_'
    pattern_cols = [col for col in df.columns if col.startswith('pattern_')]

    # If ANY pattern is flagged as 1, the master is_fraud becomes 1
    df['is_fraud'] = df[pattern_cols].max(axis=1)

    return df, pattern_cols

# --- Execution ---
# Assuming you just loaded your dataframe:
# df = pd.read_csv("/content/cleaned_dataset (4).csv")
# df.drop("Unnamed: 0", axis=1, inplace=True, errors='ignore')

# Apply the function
df, pattern_columns = engineer_fraud_features(df)

# Display the results
print("=== FINAL FRAUD COUNTS ===")
print(df['is_fraud'].value_counts())
print("\n=== BREAKDOWN BY PATTERN ===")
for col in pattern_columns:
    print(f"{col}: {df[col].sum()}")

=== FINAL FRAUD COUNTS ===
is_fraud
0    1288
1     138
Name: count, dtype: int64

=== BREAKDOWN BY PATTERN ===
pattern_ip: 103
pattern_balance: 15
pattern_fraud_ring: 6
pattern_night_owl: 4
pattern_botnet: 20
pattern_duplicate: 0
pattern_velocity: 13
